# SQL Combined Strategy Patterns

**Purpose:** Many SQL interview problems cannot be solved with a single technique. They require combining a **multi-table strategy** (choosing the right join) with a **single-table strategy** (aggregation, conditional logic, window functions). This guide teaches you to recognize these combo problems, decompose them into their component patterns, and assemble an efficient solution step by step.

**Prerequisite guides:**
- [SQL Single-Table Query Strategies](sql_single_table_query_strategies.html) — patterns for transforming data within one table
- [SQL Multi-Table Query Strategies](sql_multi_table_query_strategies.html) — patterns for combining data across tables

---

### Table of Contents

**[1. Why Combo Patterns Exist](#1-why-combo-patterns-exist)** — When one guide isn't enough

**[2. The Two-Pass Decomposition Framework](#2-two-pass-decomposition)** — How to break any combo problem into join + transform

**[3. Combo Decision Tree](#3-combo-decision-tree)** — Visual flow from problem → join type → single-table technique

**[4. Combo Pattern Library](#4-combo-pattern-library)**
- [A. LEFT JOIN + Conditional Aggregation](#combo-a) — "Metric per entity including zeros"
- [B. LEFT JOIN + COUNT with NULLs](#combo-b) — "Count per entity including zeros"
- [C. INNER JOIN + GROUP BY + HAVING](#combo-c) — "Find entities meeting a threshold in related data"
- [D. CROSS JOIN + LEFT JOIN + Conditional Aggregation](#combo-d) — "Metric per combination including zeros"
- [E. Self JOIN + Window Function](#combo-e) — "Rank or compare across self-referencing data"
- [F. LEFT JOIN + Window Function](#combo-f) — "Running totals or rankings across joined data"
- [G. Subquery/CTE + JOIN + Aggregation](#combo-g) — "Pre-filter one table, then join and summarize"
- [H. ANTI JOIN](#combo-h) — "Find entities with no match"
- [I. LEFT JOIN + Conditional Aggregation with Range/NULL Handling](#combo-i) — "Rate/Ratio with NULLs preserved"
- [J. Range JOIN + Weighted Aggregation](#combo-j) — "Weighted average with date range matching"
- [K. CROSS JOIN + LEFT JOIN + COUNT (Complete Matrix)](#combo-k) — "Zero-count entries in combinations" 

**[5. The NULL Trap — How Joins Change Single-Table Logic](#5-null-trap)** — Why CASE/COUNT/AVG behave differently after LEFT JOIN

**[6. How to Decompose Any Combo Problem](#6-decompose-any-problem)** — Step-by-step walkthrough with worked examples

**[7. Efficiency: Where to Aggregate](#7-efficiency-where-to-aggregate)** — Before the join vs after the join

**[8. Interview Script for Combo Problems](#8-interview-script)**

**[9. Common Combo Mistakes](#9-common-mistakes)**

**[10. Final Takeaway](#10-final-takeaway)**

<hr style="border: 3px solid black;">

<a id='1-why-combo-patterns-exist'></a>

## 1. Why Combo Patterns Exist

Most real SQL interview problems involve **two separate challenges packed into one question:**

| Challenge | Which Guide | Example |
|---|---|---|
| **How do I connect the tables?** | Multi-Table Guide | LEFT JOIN, CROSS JOIN, Self JOIN |
| **How do I transform the data?** | Single-Table Guide | GROUP BY, CASE WHEN, Window Functions |

When you see a problem that touches multiple tables AND requires aggregation, conditional logic, or window functions, you're looking at a **combo pattern**. Neither guide alone gives you the full answer — you need to chain them together.

### The Key Insight

> *Solve the JOIN first, then apply the single-table technique to the joined result.*

Think of it as two layers:

```
Layer 1 (Multi-Table):  Which rows do I need from which tables?  →  JOIN type
Layer 2 (Single-Table): What do I do with those rows?            →  GROUP BY / CASE / Window
```

The join determines **what rows you're working with**. The single-table technique determines **what you do with those rows**.

<hr style="border: 3px solid black;">

<a id='2-two-pass-decomposition'></a>

## 2. The Two-Pass Decomposition Framework

When you read a combo problem, make **two passes** through the question:

### Pass 1 — The Join Pass (Multi-Table Guide)

Ask these questions in order:

| # | Question | What It Tells You |
|---|---|---|
| 1 | How many tables? | Scope of the problem |
| 2 | What is each table? (dimension vs fact) | Which table is "all entities" vs "events" |
| 3 | Must the output include ALL entities, even those with no activity? | LEFT JOIN vs INNER JOIN |
| 4 | Do I need all combinations of two lists? | CROSS JOIN |
| 5 | Is a table referencing itself? | Self JOIN |

### Pass 2 — The Transform Pass (Single-Table Guide)

Now pretend the joined result is a single table and ask:

| # | Question | What It Tells You |
|---|---|---|
| 1 | Does the output have fewer rows than the joined result? | GROUP BY needed |
| 2 | Do I need to count/sum only rows meeting a condition? | CASE WHEN inside aggregate |
| 3 | Do I need to compare rows within groups? | Window function (LAG/LEAD) |
| 4 | Do I need to rank rows? | ROW_NUMBER / RANK |
| 5 | Do I need to filter groups by an aggregate condition? | HAVING |

### Pass 1 + Pass 2 = Your Solution

The join type from Pass 1 tells you how to connect tables. The technique from Pass 2 tells you how to transform the result. Chain them together and you have your query.

<hr style="border: 3px solid black;">

<a id='3-combo-decision-tree'></a>

## 3. Combo Decision Tree

Start here for any multi-table problem that also needs transformation:

<div class="fc">
  <div class="fc-node fc-start">READ THE QUESTION<br/>1. How many tables?<br/>2. Does output include ALL entities?<br/>3. What transformation is needed?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action"><a href="sql_multi_table_query_strategies.html#2-decision-tree-5-second-version" data-no-arrow>PASS 1: Pick the JOIN<br/>(Use the decision flow below →)</a></div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-b-left-join">LEFT JOIN</a></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-a-inner-join">INNER JOIN</a></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-c-cross-join">CROSS JOIN</a></div>
    </div>
  </div>
  <div style="text-align: center; margin: 10px 0;">
    <span style="display: inline-block; margin: 0 20px;">ANTI JOIN</span>
    <span style="display: inline-block;">Self JOIN</span>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action"><a href="sql_single_table_query_strategies.html#2-decision-tree-5-second-version" data-no-arrow>PASS 2: Pick the TRANSFORMATION<br/>(Treat joined result as a single table →)</a></div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-node fc-warn"><a href="sql_single_table_query_strategies.html#pattern-m-conditional-aggregation" data-no-arrow>GROUP BY + CASE<br/>(rate, ratio, conditional metric)</a><code>SUM(CASE WHEN cond THEN 1.0 ELSE 0 END) / COUNT(*)</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-warn"><a href="sql_single_table_query_strategies.html#pattern-d-combine-rows" data-no-arrow>GROUP BY + COUNT/SUM<br/>(count per entity incl zeros)</a><code>SELECT a.name, COUNT(b.id)
FROM a LEFT JOIN b ON ...
GROUP BY a.name</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-node fc-warn"><a href="sql_single_table_query_strategies.html#pattern-b-compare-rows" data-no-arrow>Window Func LAG/LEAD<br/>ROW_NUMBER<br/>(rank across joined data)</a><code>ROW_NUMBER() OVER(
  PARTITION BY grp
  ORDER BY val DESC)</code></div>
    </div>
  </div>
  <div style="text-align: center; margin: 10px 0;">
    <div class="fc-node fc-warn" style="display: inline-block; width: auto; margin-top: 10px;"><a href="sql_single_table_query_strategies.html#pattern-j-filtering-after-aggregation" data-no-arrow>GROUP BY + HAVING<br/>(filter groups by threshold)</a><code>GROUP BY col
HAVING COUNT(*) > n</code></div>
  </div>
</div>

---

### Pass 1 Expanded: How to Pick the Right JOIN

Use this flow to determine which join type your problem requires:

<div class="fc">
  <div class="fc-node fc-start">Does a table reference ITSELF?<br/>(managerId → id, parent_id → id)</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-g-self-join">SELF JOIN<br/>Same table, two aliases</a><code>FROM emp a JOIN emp b
  ON a.mgr_id = b.id</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-start">Are the two tables INDEPENDENT lists that need ALL combinations?<br/>(every student × every subject)</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-c-cross-join">CROSS JOIN<br/>Cartesian product of both lists</a><code>FROM t1 CROSS JOIN t2</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-start">Does the output need ONLY entities with NO match?<br/>("never," "didn't," "who has no," "not in")</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-warn"><a href="sql_multi_table_query_strategies.html#pattern-e-find-missing">ANTI JOIN<br/>(LEFT JOIN + WHERE IS NULL)</a><code>FROM a LEFT JOIN b ON ...
WHERE b.key IS NULL</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-start">Does the output need ALL entities, even those with no match?<br/>(zeros in output)</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-b-left-join" data-no-arrow>LEFT JOIN<br/>Keeps ALL rows; NULLs where no match</a><code>FROM a LEFT JOIN b
  ON a.id = b.fk</code></div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-good"><a href="sql_multi_table_query_strategies.html#pattern-a-inner-join" data-no-arrow>INNER JOIN<br/>Only matching rows</a><code>FROM a JOIN b ON a.id = b.fk</code></div>
    </div>
  </div>
</div>

---

### The 4 Join Types — When and Why

**LEFT JOIN** — *"Keep everyone, even those with nothing"*

| Use When | What It Does | NULL Trap |
|---|---|---|
| Output includes ALL entities from one table, even those with zero activity | Returns all rows from the left table; fills right-side columns with NULL when no match | `COUNT(*)` counts NULL rows as 1; use `COUNT(right_col)` instead |

**Signals in the problem:** "including those with zero," "even if no orders," "all users," expected output shows `0` or `0.00` for some rows.

**Example scenario:** "Find the confirmation rate for each user, including those who never requested a confirmation." User 6 has no confirmations but still appears with 0.00.

```
Signups (ALL users)  ──LEFT JOIN──▶  Confirmations (events)
   4 users                              only 3 users have events
   Result: 4 rows                       user 6 gets NULLs → handle with CASE or COUNT(col)
```

---

**INNER JOIN** — *"Only what matches"*

| Use When | What It Does | NULL Trap |
|---|---|---|
| You only need rows that exist in BOTH tables | Returns only rows with a match on both sides; no NULLs from the join itself | None — unmatched rows are simply dropped |

**Signals in the problem:** "find," "get the name of," "look up," no zeros in expected output, only entities with activity appear.

**Example scenario:** "Find managers with at least 5 direct reports." You only care about managers who HAVE reports — those without any are irrelevant to the answer.

```
Employee (managers)  ──INNER JOIN──▶  Employee (reports)
   only managers WITH reports survive the join
   managers with 0 reports are dropped (which is what we want)
```

---

**CROSS JOIN** — *"Every possible combination"*

| Use When | What It Does | NULL Trap |
|---|---|---|
| The output needs every row from table A paired with every row from table B | Cartesian product: if A has 4 rows and B has 3, result has 12 rows | None from CROSS JOIN itself, but usually followed by LEFT JOIN which introduces NULLs |

**Signals in the problem:** "every student with every subject," "all possible pairs," two independent dimension tables with no direct relationship.

**Example scenario:** "Count how many times each student attended each exam." Students and Subjects are independent lists — you need all 12 combinations, then LEFT JOIN to Examinations for the actual counts.

```
Students (4 rows)  ──CROSS JOIN──▶  Subjects (3 rows)  =  12 combinations
   then LEFT JOIN to Examinations to attach actual exam data
   combinations with no exams get COUNT = 0
```

**Key rule:** CROSS JOIN is only for two **independent lists** that have no direct foreign key relationship. If the tables ARE related (foreign key), you need LEFT or INNER JOIN instead.

---

**ANTI JOIN (LEFT JOIN + WHERE IS NULL)** — *"Find what's missing"*

| Use When | What It Does | NULL Trap |
|---|---|---|
| Output needs only entities with NO match in another table | LEFT JOIN then filter where the right side is NULL — keeps only the unmatched rows | No trap — you're explicitly using the NULLs as your filter |

**Signals in the problem:** "never ordered," "didn't attend," "customers with no," "who has not," "find missing."

**Example scenario:** "Find customers who have never placed an order." You don't want all customers with a count — you want ONLY the ones with zero orders.

```
Customers  ──LEFT JOIN──▶  Orders  ──WHERE o.id IS NULL──▶  Only unmatched customers
   4 customers                 3 have orders
   Result: 1 customer (the one with NO orders)
```

**ANTI JOIN vs LEFT JOIN — how to tell them apart:** Both start with LEFT JOIN, but the intent is different. If the output needs ALL entities with a metric (including zeros), that's a LEFT JOIN. If the output needs ONLY the entities with nothing, that's an ANTI JOIN.

| Output Shows | Pattern |
|---|---|
| All entities, some with 0 | LEFT JOIN (keep all, aggregate) |
| Only entities with nothing | ANTI JOIN (LEFT JOIN + WHERE IS NULL) |

**Three ways to write an ANTI JOIN (ranked):**

| Rank | Method | Why |
|---|---|---|
| 1 | `LEFT JOIN` + `WHERE right_col IS NULL` | Clear intent, efficient, NULL-safe |
| 2 | `NOT EXISTS (SELECT 1 FROM ...)` | Also efficient, short-circuits, NULL-safe |
| 3 | `NOT IN (SELECT col FROM ...)` | Breaks silently if subquery returns NULLs — avoid |

---

**Self JOIN** — *"One table, two roles"*

| Use When | What It Does | NULL Trap |
|---|---|---|
| A table has a column that references its own primary key (parent/child, manager/employee) | Joins the table to itself using two aliases, each representing a different role | Use LEFT self-join if some entities have no parent (e.g., CEO has no manager) |

**Signals in the problem:** "manager name," "reports to," "parent category," single table with a column like `managerId`, `parent_id`, `supervisor_id`.

**Example scenario:** "Find each employee's manager name." The Employee table has both the employee and manager info — you just need to match `managerId` to `id`.

```
Employee AS e (employee role)  ──JOIN──▶  Employee AS m (manager role)
   e.managerId = m.id
   Same table, two different perspectives
```

---

### How to Tell LEFT JOIN from INNER JOIN — The Zero Test

The single most important signal is: **does the expected output contain zeros or entities with no activity?**

| What You See in Expected Output | Join Type |
|---|---|
| Every entity appears, some with 0 or NULL values | **LEFT JOIN** |
| Only entities with NO activity appear | **ANTI JOIN** (LEFT JOIN + WHERE IS NULL) |
| Only entities with activity appear | **INNER JOIN** |
| All combinations of two lists, some with 0 | **CROSS JOIN** + **LEFT JOIN** |
| One table, column references its own PK | **Self JOIN** |

**The Zero Test:** If you see a `0`, `0.00`, or `NULL` for an entity that has no matching rows in the other table, that's a LEFT JOIN. If such entities are simply absent from the output, that's an INNER JOIN.

---

### Quick Pattern Matcher

| If you see... | Pass 1 (Join) | Pass 2 (Transform) | Combo Pattern |
|---|---|---|---|
| "rate/ratio per entity, including those with none" | LEFT JOIN | GROUP BY + CASE | **A** |
| "count per entity including zeros" | LEFT JOIN | GROUP BY + COUNT(column) | **B** |
| "entities meeting a threshold in related data" | INNER JOIN | GROUP BY + HAVING | **C** |
| "metric per combination including zeros" | CROSS JOIN + LEFT JOIN | GROUP BY + CASE or COUNT | **D** |
| "rank within self-referencing hierarchy" | Self JOIN | Window Function | **E** |
| "running total across joined data" | LEFT JOIN | SUM() OVER() | **F** |
| "pre-filter, then join and summarize" | Subquery + JOIN | GROUP BY | **G** |
| "find entities with NO match / never / didn't" | ANTI JOIN | Often none needed (just SELECT) | **H** |

<hr style="border: 3px solid black;">

<a id='4-combo-pattern-library'></a>

## 4. Combo Pattern Library

<a id='combo-a'></a>

### A. LEFT JOIN + Conditional Aggregation — "Metric per entity including zeros"

**Signals:** "rate," "ratio," "percentage per user including those with none," "confirmation rate," "approval rate," "average score including students who didn't take the test"

---

**The Problem Shape**

You have a dimension table (all entities) and a fact table (events). The output needs:
- One row per entity (even those with zero events)
- A calculated metric (rate, ratio, average) based on conditional counting

**Pass 1 (Join):** Output shows entities with zero → must preserve all entities → **LEFT JOIN**

**Pass 2 (Transform):** Need a rate = conditional count / total count → **GROUP BY + CASE WHEN inside AVG or COUNT**

---

**Best approach — LEFT JOIN + AVG(CASE WHEN)**

**Method:** LEFT JOIN the dimension to the fact table, then GROUP BY the entity. Use `AVG(CASE WHEN condition THEN 1.0 ELSE 0.0 END)` to compute the rate in one expression. Rows with no match get NULL from the LEFT JOIN, and AVG handles this correctly.

**In plain language:** "Join all users to their confirmations (keeping users with none). Then for each user, average a 1 for confirmed and 0 for everything else — that average IS the confirmation rate."

```sql
-- Confirmation rate per user (including 0 for users with no confirmations)
SELECT
    s.user_id,
    ROUND(AVG(CASE WHEN c.action = 'confirmed' THEN 1.0 ELSE 0.0 END)::numeric, 2) AS confirmation_rate
FROM Signups AS s
LEFT JOIN Confirmations AS c
    ON s.user_id = c.user_id
GROUP BY s.user_id;
```

**Why this works for zeros:** Users with no confirmations get one row from the LEFT JOIN where `c.action` is NULL. The CASE falls to ELSE 0.0, and AVG of a single 0.0 is 0.00.

---

**Also valid — Pre-aggregate in subquery, then LEFT JOIN**

**Method:** Compute the metric in a subquery first, then LEFT JOIN the dimension table to the pre-aggregated result. Use COALESCE to handle NULLs.

**In plain language:** "First calculate each user's rate in a subquery. Then join all users to those results — users missing from the subquery get COALESCE'd to 0."

```sql
SELECT
    s.user_id,
    ROUND(COALESCE(c.confirmation_rate, 0)::numeric, 2) AS confirmation_rate
FROM Signups AS s
LEFT JOIN (
    SELECT
        user_id,
        COUNT(CASE WHEN action = 'confirmed' THEN 1 END)::numeric / COUNT(*)::numeric AS confirmation_rate
    FROM Confirmations
    GROUP BY user_id
) AS c
    ON s.user_id = c.user_id;
```

**Trade-off:** This approach separates the aggregation logic from the join, which can be clearer to read. Performance is similar — the optimizer handles both well.

---

**Efficiency ranking:**

| Rank | Method | Why |
|---|---|---|
| 1 | LEFT JOIN + AVG(CASE WHEN) | Single pass, compact, NULL-safe by design |
| 2 | Pre-aggregate in subquery + LEFT JOIN + COALESCE | Equally efficient, cleaner separation, needs explicit NULL handling |

<hr style="border: 2px solid black;">

<a id='combo-b'></a>

### B. LEFT JOIN + COUNT with NULLs — "Count per entity including zeros"

**Signals:** "how many orders per customer including those with zero," "number of posts per user," "count including none"

---

**The Problem Shape**

Same as Pattern A, but simpler — you just need a count, not a ratio. The trap is that `COUNT(*)` counts NULLs (giving 1 instead of 0 for unmatched rows).

**Pass 1 (Join):** Preserve all entities → **LEFT JOIN**

**Pass 2 (Transform):** Count events per entity → **GROUP BY + COUNT(specific_column)**

---

**Best approach — LEFT JOIN + COUNT(fact_column)**

**Method:** Use `COUNT(column_from_right_table)` — not `COUNT(*)`. `COUNT(column)` skips NULLs, so unmatched rows correctly get 0.

**In plain language:** "Join all customers to their orders. Count the order_id column — NULLs from the LEFT JOIN are skipped, giving 0 for customers with no orders."

```sql
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(o.order_id) AS order_count
FROM Customers AS c
LEFT JOIN Orders AS o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name;
```

**Critical trap:** If you use `COUNT(*)` instead of `COUNT(o.order_id)`, customers with no orders get 1 (because the LEFT JOIN still produces one row with NULLs, and `COUNT(*)` counts that row).

---

**The COUNT(*) vs COUNT(column) Rule**

| Expression | Counts NULLs? | After LEFT JOIN with no match |
|---|---|---|
| `COUNT(*)` | Yes | Returns **1** (wrong!) |
| `COUNT(o.order_id)` | No | Returns **0** (correct!) |

**Memory rule:** After a LEFT JOIN, always COUNT a column from the **right** (joined) table, never `COUNT(*)`.

<hr style="border: 2px solid black;">

<a id='combo-c'></a>

### C. INNER JOIN + GROUP BY + HAVING — "Find entities meeting a threshold in related data"

**Signals:** "managers with at least 5 reports," "departments with more than 10 employees," "categories with average price above X"

---

**The Problem Shape**

You need to filter entities based on an aggregate condition on related data. The output only includes entities that meet the threshold — no need to preserve unmatched rows.

**Pass 1 (Join):** Only need matching rows → **INNER JOIN** (or Self JOIN if same table)

**Pass 2 (Transform):** Filter by aggregate → **GROUP BY + HAVING**

---

**Best approach — Self JOIN + GROUP BY + HAVING (when same table)**

**Method:** When the "related data" is in the same table (like employees and their managers), join the table to itself, then aggregate and filter.

**In plain language:** "Pair each manager with their reports, count the reports per manager, keep only those with 5 or more."

```sql
-- Managers with at least 5 direct reports
SELECT m.name
FROM Employee m
JOIN Employee r
    ON m.id = r.managerId
GROUP BY m.id, m.name
HAVING COUNT(*) >= 5;
```

---

**Also valid — INNER JOIN across tables + GROUP BY + HAVING**

**Method:** When the threshold is based on a different table, INNER JOIN to that table, then aggregate and filter.

```sql
-- Departments with more than 10 employees
SELECT d.department_name, COUNT(*) AS emp_count
FROM Departments d
INNER JOIN Employees e
    ON d.department_id = e.department_id
GROUP BY d.department_id, d.department_name
HAVING COUNT(*) > 10;
```

---

**Efficiency ranking:**

| Rank | Method | Why |
|---|---|---|
| 1 | JOIN + GROUP BY + HAVING | Direct, single query |
| 2 | Pre-aggregate in CTE/subquery + JOIN | Equal performance, cleaner separation |
| 3 | Correlated subquery in WHERE | Runs per row — slower |

<hr style="border: 2px solid black;">

<a id='combo-d'></a>

### D. CROSS JOIN + LEFT JOIN + Conditional Aggregation — "Metric per combination including zeros"

**Signals:** "how many times each student attended each exam," "score per student per subject including zeros," "rate per user per category"

---

**The Problem Shape**

You have two dimension tables and one fact table. The output needs every possible combination of the two dimensions, with a metric (count, average, rate) for each — including zero where no fact rows exist.

**Pass 1 (Join):** Need all combinations of two lists → **CROSS JOIN** dimensions, then **LEFT JOIN** to fact table

**Pass 2 (Transform):** Aggregate per combination → **GROUP BY + COUNT(column) or CASE WHEN**

---

**Best approach — CROSS JOIN + LEFT JOIN + COUNT(fact_column)**

```sql
-- Count exams per student per subject (including 0)
SELECT
    s.student_id,
    s.student_name,
    sub.subject_name,
    COUNT(e.student_id) AS attended_exams
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN Examinations AS e
    ON e.student_id = s.student_id
    AND e.subject_name = sub.subject_name
GROUP BY s.student_id, s.student_name, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**Why COUNT(e.student_id):** Same NULL trick as Pattern B — `COUNT(column)` skips NULLs from unmatched LEFT JOIN rows, giving 0 instead of 1.

---

**Also valid — Pre-aggregate fact table, then CROSS JOIN + LEFT JOIN + COALESCE**

```sql
SELECT
    s.student_id,
    s.student_name,
    sub.subject_name,
    COALESCE(e.attended_exams, 0) AS attended_exams
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN (
    SELECT student_id, subject_name, COUNT(*) AS attended_exams
    FROM Examinations
    GROUP BY student_id, subject_name
) AS e
    ON e.student_id = s.student_id
    AND e.subject_name = sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**When to pre-aggregate:** When the fact table is very large. Pre-aggregating reduces its row count before the LEFT JOIN.

<hr style="border: 2px solid black;">

<a id='combo-e'></a>

### E. Self JOIN + Window Function — "Rank or compare across self-referencing data"

**Signals:** "rank employees within each manager's team," "compare each employee's salary to their department average," "top performer per manager"

---

**The Problem Shape**

A self-referencing table (e.g., employees with managerId) where you need to apply a window function across groups defined by the self-reference.

**Pass 1 (Join):** Table references itself → **Self JOIN**

**Pass 2 (Transform):** Rank or compare within groups → **Window Function**

---

**Example — Top earner per manager**

```sql
SELECT manager_name, employee_name, salary
FROM (
    SELECT
        m.name AS manager_name,
        e.name AS employee_name,
        e.salary,
        ROW_NUMBER() OVER (PARTITION BY e.managerId ORDER BY e.salary DESC) AS rn
    FROM Employee e
    INNER JOIN Employee m
        ON e.managerId = m.id
) ranked
WHERE rn = 1;
```

**Decomposition:** Self JOIN pairs each employee with their manager's name. Then ROW_NUMBER ranks employees within each manager's group by salary. Filter to rn = 1 keeps only the top earner.

<hr style="border: 2px solid black;">

<a id='combo-f'></a>

### F. LEFT JOIN + Window Function — "Running totals or rankings across joined data"

**Signals:** "cumulative orders per customer over time," "rank products by sales within each category," "running balance including months with no transactions"

---

**The Problem Shape**

You need to join tables first (to get all entities or enrich data), then apply a window function to the joined result.

**Pass 1 (Join):** Preserve all entities or enrich → **LEFT JOIN** or **INNER JOIN**

**Pass 2 (Transform):** Running total, ranking, or row comparison → **Window Function**

---

**Example — Cumulative order total per customer**

```sql
SELECT
    c.customer_id,
    c.customer_name,
    o.order_date,
    o.amount,
    SUM(o.amount) OVER (PARTITION BY c.customer_id ORDER BY o.order_date) AS running_total
FROM Customers c
INNER JOIN Orders o
    ON c.customer_id = o.customer_id;
```

**Decomposition:** INNER JOIN brings in customer names. SUM() OVER() computes the cumulative total within each customer's orders.

<hr style="border: 2px solid black;">

<a id='combo-g'></a>

### G. Subquery/CTE + JOIN + Aggregation — "Pre-filter one table, then join and summarize"

**Signals:** "only active users' total orders," "average salary of employees in departments with more than 5 people," "recent orders from premium customers"

---

**The Problem Shape**

You need to filter or transform one table first (using a single-table technique), then join the result to another table for further work.

**Pass 1 (Transform first):** Apply a single-table technique in a CTE/subquery

**Pass 2 (Join):** Join the CTE result to another table

---

**Example — Average salary in large departments**

```sql
WITH large_depts AS (
    SELECT department_id
    FROM Employees
    GROUP BY department_id
    HAVING COUNT(*) > 5
)
SELECT
    d.department_name,
    ROUND(AVG(e.salary)::numeric, 2) AS avg_salary
FROM large_depts ld
INNER JOIN Departments d
    ON ld.department_id = d.department_id
INNER JOIN Employees e
    ON ld.department_id = e.department_id
GROUP BY d.department_id, d.department_name;
```

**Decomposition:** CTE pre-filters to departments with >5 employees (single-table: GROUP BY + HAVING). Then JOIN to Departments for names and Employees for salary data. Final GROUP BY computes the average.

**When to use this pattern:** When one table needs significant pre-processing before it's useful in a join. The CTE isolates that logic cleanly.

<hr style="border: 2px solid black;">

<a id='combo-h'></a>

### H. ANTI JOIN — "Find entities with no match"

**Signals:** "never ordered," "customers with no," "who didn't attend," "find missing," "not in"

---

**The Problem Shape**

You need to find entities in one table that have NO corresponding rows in another table. Unlike LEFT JOIN patterns (A and B) where you keep everyone and show zeros, here you ONLY want the unmatched entities.

**Pass 1 (Join):** Need only unmatched rows → **ANTI JOIN** (LEFT JOIN + WHERE IS NULL)

**Pass 2 (Transform):** Often none — you just need the list. But sometimes you aggregate the unmatched entities further (count them, group by category, etc.)

---

**Best approach — LEFT JOIN + WHERE IS NULL**

**Method:** LEFT JOIN to the fact table, then filter where the fact table's key column IS NULL. This keeps only the rows from the left table that had no match.

**In plain language:** "Join all customers to their orders. Customers with no orders will have NULL in the order columns. Keep only those NULLs — those are the customers who never ordered."

```sql
-- Customers who never placed an order
SELECT c.customer_id, c.customer_name
FROM Customers c
LEFT JOIN Orders o
    ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;
```

---

**Also strong — NOT EXISTS**

**Method:** For each row in the main table, check if any matching row exists in the other table. If not, keep it. Short-circuits after finding the first match.

```sql
SELECT c.customer_id, c.customer_name
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1 FROM Orders o
    WHERE o.customer_id = c.customer_id
);
```

---

**Avoid — NOT IN (when NULLs are possible)**

```sql
-- DANGEROUS: Returns 0 rows if any customer_id in Orders is NULL
SELECT customer_id, customer_name
FROM Customers
WHERE customer_id NOT IN (SELECT customer_id FROM Orders);
```

---

**ANTI JOIN + further aggregation (combo within a combo)**

Sometimes you need to find what's missing AND then summarize it:

```sql
-- Count how many customers per region have never ordered
SELECT c.region, COUNT(*) AS inactive_customers
FROM Customers c
LEFT JOIN Orders o
    ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
GROUP BY c.region;
```

**Decomposition:** ANTI JOIN finds the unmatched customers. GROUP BY + COUNT summarizes them by region. This is a combo of an anti-join pattern with a single-table aggregation pattern.

---

**Efficiency ranking:**

| Rank | Method | Why |
|---|---|---|
| 1 | LEFT JOIN + WHERE IS NULL | Clear intent, efficient, NULL-safe |
| 2 | NOT EXISTS | Also efficient, short-circuits, NULL-safe |
| 3 | NOT IN | Breaks with NULLs — avoid unless column is guaranteed NOT NULL |

<hr style="border: 2px solid black;">

<a id='combo-i'></a>

### I. LEFT JOIN + Conditional Aggregation with Range/NULL Handling — "Rate/Ratio with NULLs preserved"

**Signals:** "rate including users with no activity," "approval percentage with nulls," "confirmation rate where nulls count as zero"

---

**The Problem Shape**

This is an advanced version of Pattern A where you must preserve NULL values in your aggregation, and WHERE conditions can be applied to handle optional records. A classic scenario: LEFT JOIN a dimension table to a fact table, and count events conditional on both a status AND whether the record is NULL.

**Pass 1 (Join):** Output preserves all entities even with no matches → **LEFT JOIN**

**Pass 2 (Transform):** Conditional count/ratio, handling NULLs explicitly → **GROUP BY + SUM(CASE WHEN)/COUNT + WHERE conditions that check for NULL**

---

**Example 1: Confirmation Rate (#1934) — LEFT JOIN + AVG(CASE)**

**Problem:** Find the confirmation rate for each user, including those with zero confirmations. Unconfirmed actions and NULLs should both count as "not confirmed."

**Method:** LEFT JOIN Signups to Confirmations (preserving users with zero). For each user, average a 1 for confirmed actions, 0 for everything else (including NULLs from the LEFT JOIN).

```sql
-- Confirmation rate per user (including 0 for users with no confirmations)
SELECT
    s.user_id,
    ROUND(AVG(CASE WHEN c.action = 'confirmed' THEN 1.0 ELSE 0.0 END), 2) AS confirmation_rate
FROM Signups AS s
LEFT JOIN Confirmations AS c
    ON s.user_id = c.user_id
GROUP BY s.user_id
ORDER BY s.user_id;
```

**Why this works:**
- Users with zero confirmations still have one row (all columns NULL on the right side)
- CASE WHEN c.action = 'confirmed' evaluates to FALSE (NULL never equals anything), so ELSE 0.0 is used
- AVG(0.0) = 0.00 ✓
- Users WITH confirmations have multiple rows, some with 'confirmed', some with other actions
- AVG correctly averages the 1s and 0s across all rows per user

**Key trap:** COUNT(*) instead of AVG(CASE) would give 1 for unmatched users (one NULL row), not 0. Use AVG(CASE) to compute the true rate.

---

**Example 2: Employee Bonus (#577) — LEFT JOIN with WHERE conditions on NULLs**

**Problem:** Find employees whose bonus is less than 1000 OR who received no bonus at all. Result should show all such employees.

**Method:** LEFT JOIN Employee to Bonus. Use WHERE to filter for records where bonus IS NULL OR bonus < 1000.

```sql
-- Employees with no bonus or bonus < 1000
SELECT
    e.employee_id,
    e.name,
    b.bonus
FROM Employee AS e
LEFT JOIN Bonus AS b
    ON e.employee_id = b.employee_id
WHERE b.bonus < 1000 OR b.bonus IS NULL
ORDER BY e.employee_id;
```

**Why this works:**
- The LEFT JOIN returns all employees, with NULL bonus for those who have no bonus record
- WHERE b.bonus IS NULL catches those with no bonus (NULL is not < 1000, so we need the OR)
- WHERE b.bonus < 1000 catches those with an explicit bonus record under 1000
- Together, this filters to exactly the employees we need

**Key trap:** Using just WHERE b.bonus < 1000 would exclude NULL rows (since NULL < 1000 is unknown, not true). Must use OR b.bonus IS NULL.

---

**The COUNT(*) vs COUNT(column) Rule (Revisited)**

After a LEFT JOIN, the NULL-handling rule becomes critical:

| Expression | Result After LEFT JOIN with No Match | Use Case |
|---|---|---|
| `COUNT(*)` | Returns **1** (counts the NULL row) | Never after LEFT JOIN |
| `COUNT(right_col)` | Returns **0** (skips NULL) | Count matched records |
| `AVG(CASE WHEN cond THEN 1 ELSE 0 END)` | Returns **0.0** (correct rate) | Calculate rate/ratio |
| `SUM(CASE WHEN cond THEN amount ELSE 0 END)` | Returns **0** (sums zeros) | Sum with conditions |

**Memory rule:** After LEFT JOIN, always be explicit: use COUNT(column) for counts, CASE for conditions, and never rely on COUNT(*).

<hr style="border: 2px solid black;">

<a id='combo-j'></a>

### J. Range JOIN + Weighted Aggregation — "Weighted average with date range matching"

**Signals:** "average selling price," "weighted average," "price on the date of purchase," "lookup by range," "BETWEEN date range"

---

**The Problem Shape**

You need to JOIN two tables not on equality, but on a **range condition** (typically a date range). Then compute a weighted aggregate from the joined result.

Classic pattern: One table has "events" with dates, another has "price lists" with start/end dates. You join each event to the price list entry valid on that event's date, then compute a weighted average.

**Pass 1 (Join):** Match on a range condition (BETWEEN) → **JOIN ... ON column BETWEEN start AND end**

**Pass 2 (Transform):** Compute weighted average → **GROUP BY + SUM(price * units) / SUM(units)**

---

**Example: Average Selling Price (#1251) — Range JOIN + Weighted Avg**

**Problem:** Given UnitsSold (product_id, units, purchase_date) and Prices (product_id, price, start_date, end_date), find the average selling price for each product. Price is valid for the date range [start_date, end_date].

**Method:**
1. JOIN UnitsSold to Prices on product_id AND purchase_date BETWEEN start_date AND end_date
2. GROUP BY product_id
3. Compute weighted average: SUM(price * units) / SUM(units)

```sql
-- Average selling price per product, using the price valid on the purchase date
SELECT
    u.product_id,
    ROUND(SUM(u.units * p.price) / SUM(u.units)::numeric, 2) AS average_price
FROM UnitsSold AS u
JOIN Prices AS p
    ON u.product_id = p.product_id
    AND u.purchase_date BETWEEN p.start_date AND p.end_date
GROUP BY u.product_id
ORDER BY u.product_id;
```

**Why this works:**
- The JOIN condition has TWO parts: equality (product_id) and range (BETWEEN)
- Each UnitsSold row joins to the SINGLE Prices row valid on that date
- SUM(price * units) / SUM(units) is the weighted average (sum of weighted values / count of weights)

**Key insight:** Weighted average = (sum of values × weights) / (sum of weights). Here, values = price, weights = units.

---

**Critical Trap: Products with Zero Sales**

The query above uses INNER JOIN, so products with no sales don't appear. If you need ALL products (showing 0 price for those with no sales), use LEFT JOIN from a Prices or Products table:

```sql
-- All products, including those with no sales (price = 0)
SELECT
    p.product_id,
    COALESCE(ROUND(SUM(u.units * pr.price) / SUM(u.units)::numeric, 2), 0) AS average_price
FROM (
    SELECT DISTINCT product_id FROM Prices
) AS p
LEFT JOIN UnitsSold AS u
    ON u.product_id = p.product_id
LEFT JOIN Prices AS pr
    ON u.product_id = pr.product_id
    AND u.purchase_date BETWEEN pr.start_date AND pr.end_date
GROUP BY p.product_id
ORDER BY p.product_id;
```

**Key change:** LEFT JOIN from all products ensures no product is dropped, and COALESCE handles the NULL result for products with zero units sold.

---

**Range JOIN Variants**

| Condition | Example |
|---|---|
| Date range | `purchase_date BETWEEN start_date AND end_date` |
| Numeric range | `value BETWEEN min_val AND max_val` |
| Multiple ranges | `date BETWEEN start_date AND end_date AND value BETWEEN min_val AND max_val` |
| Inequality | `purchase_date >= start_date AND purchase_date < end_date` (inclusive of start, exclusive of end) |

---

**Common Mistake: Missing the Range in the JOIN**

Wrong:
```sql
-- This joins every UnitsSold to EVERY Prices row (Cartesian product)
FROM UnitsSold u
JOIN Prices p ON u.product_id = p.product_id
```

Right:
```sql
-- This joins each UnitsSold to only the Prices row valid on that date
FROM UnitsSold u
JOIN Prices p
    ON u.product_id = p.product_id
    AND u.purchase_date BETWEEN p.start_date AND p.end_date
```

Without the BETWEEN, you get multiple price matches per unit, corrupting the weighted average.

<hr style="border: 2px solid black;">

<a id='combo-k'></a>

### K. CROSS JOIN + LEFT JOIN + COUNT (Complete Matrix) — "Zero-count entries in combinations"

**Signals:** "every student × every subject," "all possible combinations," "students and exams," "count attendance for all possible pairs"

---

**The Problem Shape**

You have two independent dimension tables (students, subjects) and a fact table (exam attendance). The output must include ALL combinations of students and subjects, even those where the student never took that exam (count = 0).

This is a **complete matrix** pattern: use CROSS JOIN to generate all combinations, then LEFT JOIN to the fact table to attach actual data.

**Pass 1 (Join):** Generate all combinations, then match to facts, preserving unmatched combos → **CROSS JOIN, then LEFT JOIN**

**Pass 2 (Transform):** Count matched rows per combination → **GROUP BY all dimensions + COUNT(fact_column)**

---

**Example: Students and Examinations (#1280) — Complete Matrix**

**Problem:** Given Students (student_id), Subjects (subject_name), and Examinations (student_id, subject_name, exam_date), find the count of exams each student took for each subject. Include all student-subject combinations, even those with zero exams.

**Method:**
1. CROSS JOIN Students × Subjects to create all combinations
2. LEFT JOIN to Examinations to attach actual exam data
3. GROUP BY student and subject
4. COUNT(exam_id) — only counts matched exams, gives 0 for unmatched combinations

```sql
-- Count exams per student-subject combination (including zeros)
SELECT
    s.student_id,
    sub.subject_name,
    COUNT(e.exam_id) AS attended_exams
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN Examinations AS e
    ON s.student_id = e.student_id
    AND sub.subject_name = e.subject_name
GROUP BY s.student_id, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**Why this works:**
- CROSS JOIN produces all 4 × 3 = 12 combinations (4 students × 3 subjects)
- LEFT JOIN attaches exam data where it exists; unmatched combos get NULL in exam columns
- GROUP BY all dimensions ensures one row per combination
- COUNT(e.exam_id) skips NULLs, so unmatched combos get 0 ✓

**Key insight:** The CROSS JOIN + LEFT JOIN pattern is essential when you need "zero entries" for missing combinations. Without it (using only INNER JOIN), unmatched combos simply disappear.

---

**Why Not Just COUNT(*)?**

After CROSS JOIN + LEFT JOIN, you have one row per combination (with NULLs for unmatched). Don't use COUNT(*) — it would always return 1 (the one row exists, even if all right-side columns are NULL).

| Expression | Result |
|---|---|
| `COUNT(*)` | **1** (counts the row itself, regardless of NULLs) |
| `COUNT(e.exam_id)` | **0** (skips NULL, only counts matched exams) |

**Memory rule:** After LEFT JOIN, use COUNT(column), not COUNT(*).

---

**Variant: What If You Also Need a SUM or AVG?**

If you need to aggregate a value (e.g., average score) instead of just counting:

```sql
-- Count exams and average score per student-subject combination
SELECT
    s.student_id,
    sub.subject_name,
    COUNT(e.exam_id) AS attended_exams,
    ROUND(AVG(COALESCE(e.score, 0)), 2) AS avg_score
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN Examinations AS e
    ON s.student_id = e.student_id
    AND sub.subject_name = e.subject_name
GROUP BY s.student_id, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

Use COALESCE(e.score, 0) to treat missing exams as 0 score (or use AVG(e.score) to count only actual exams).

---

**Common Mistake: Forgetting the CROSS JOIN**

Wrong (only returns combos with at least one exam):
```sql
-- Missing combinations are dropped
SELECT
    s.student_id,
    e.subject_name,
    COUNT(e.exam_id) AS attended_exams
FROM Students AS s
JOIN Examinations AS e ON s.student_id = e.student_id
GROUP BY s.student_id, e.subject_name;
```

Right (includes all combos, even those with zero exams):
```sql
-- All student-subject combinations are preserved
SELECT
    s.student_id,
    sub.subject_name,
    COUNT(e.exam_id) AS attended_exams
FROM Students AS s
CROSS JOIN Subjects AS sub
LEFT JOIN Examinations AS e
    ON s.student_id = e.student_id
    AND sub.subject_name = e.subject_name
GROUP BY s.student_id, sub.subject_name;
```


<hr style="border: 3px solid black;">

<a id='5-null-trap'></a>

## 5. The NULL Trap — How Joins Change Single-Table Logic

The most common mistake in combo problems: **single-table functions behave differently when LEFT JOIN introduces NULLs.**

### The Problem

When a LEFT JOIN finds no match, every column from the right table becomes NULL. Functions that work perfectly on a single table can give wrong results on these NULLs.

### NULL Behavior Reference

| Function | With NULLs from LEFT JOIN | Result | Fix |
|---|---|---|---|
| `COUNT(*)` | Counts the NULL row | **1 instead of 0** | Use `COUNT(right_table.column)` |
| `COUNT(column)` | Skips NULLs | **0** (correct) | No fix needed |
| `SUM(column)` | Skips NULLs | **NULL** | Wrap in `COALESCE(SUM(...), 0)` |
| `AVG(column)` | Skips NULLs | **NULL** | Wrap in `COALESCE(AVG(...), 0)` |
| `AVG(CASE WHEN ... THEN 1.0 ELSE 0.0 END)` | ELSE catches NULLs as 0 | **0.0** (correct) | No fix needed |
| `MAX(column)` / `MIN(column)` | Skips NULLs | **NULL** | Wrap in `COALESCE(...)` if needed |
| `CASE WHEN col = 'x'` | NULL ≠ 'x', falls to ELSE | Depends on ELSE | Ensure ELSE handles the NULL case |

### The Golden Rule

> *After a LEFT JOIN, never use `COUNT(*)` — always `COUNT(column_from_right_table)`. For SUM and AVG, wrap in `COALESCE` or use CASE with an ELSE that handles the NULL path.*

### Example: The COUNT(*) Trap

```sql
-- WRONG: Customers with no orders get order_count = 1
SELECT c.customer_id, COUNT(*) AS order_count
FROM Customers c
LEFT JOIN Orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;

-- RIGHT: Customers with no orders get order_count = 0
SELECT c.customer_id, COUNT(o.order_id) AS order_count
FROM Customers c
LEFT JOIN Orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;
```

<hr style="border: 3px solid black;">

<a id='6-decompose-any-problem'></a>

## 6. How to Decompose Any Combo Problem

Use this step-by-step process for **any** problem that involves multiple tables and a transformation:

### Step 1 — List the Tables and Classify Them

> *"I have [N] tables. [X] is a dimension (entities), [Y] is a fact (events)."*

### Step 2 — Look at the Expected Output

Ask:
- Does every entity appear (including those with no events)? → LEFT JOIN
- Do I see zeros in the output? → LEFT JOIN + careful NULL handling
- Is it one row per entity? → GROUP BY needed
- Is it a rate or ratio? → CASE WHEN inside AVG or COUNT

### Step 3 — Pick the Join (Pass 1)

Use the multi-table decision tree.

### Step 4 — Pick the Transform (Pass 2)

Pretend the joined result is a single table. Use the single-table decision tree.

### Step 5 — Handle NULLs

Check: did your join introduce NULLs? If yes, review the NULL Trap table in Section 5.

---

### Worked Example 1: Confirmation Rate

**Tables:** Signups (dimension — all users), Confirmations (fact — confirmation attempts)

| Step | Answer |
|---|---|
| Classify tables | Signups = dimension, Confirmations = fact |
| Output includes all entities? | Yes — user 6 has 0.00 rate → must preserve all users |
| Join type (Pass 1) | **LEFT JOIN** from Signups to Confirmations |
| Output shape | One row per user (collapsing) → GROUP BY |
| Transformation needed | Rate = confirmed / total → CASE WHEN inside AVG |
| NULL handling | LEFT JOIN NULLs → CASE ELSE 0.0 handles it |
| **Combo Pattern** | **A: LEFT JOIN + Conditional Aggregation** |

```sql
SELECT
    s.user_id,
    ROUND(AVG(CASE WHEN c.action = 'confirmed' THEN 1.0 ELSE 0.0 END)::numeric, 2) AS confirmation_rate
FROM Signups AS s
LEFT JOIN Confirmations AS c
    ON s.user_id = c.user_id
GROUP BY s.user_id;
```

---

### Worked Example 2: Students × Subjects × Examinations

**Tables:** Students (dimension), Subjects (dimension), Examinations (fact)

| Step | Answer |
|---|---|
| Classify tables | Students = dimension, Subjects = dimension, Examinations = fact |
| Output includes all combinations? | Yes — every student × every subject, even with 0 |
| Join type (Pass 1) | **CROSS JOIN** Students × Subjects, then **LEFT JOIN** Examinations |
| Output shape | One row per student-subject pair → GROUP BY |
| Transformation needed | Count of exams → COUNT(fact_column) |
| NULL handling | LEFT JOIN NULLs → COUNT(e.student_id) skips NULLs = 0 |
| **Combo Pattern** | **D: CROSS JOIN + LEFT JOIN + COUNT(column)** |

```sql
SELECT
    s.student_id, s.student_name, sub.subject_name,
    COUNT(e.student_id) AS attended_exams
FROM Students s
CROSS JOIN Subjects sub
LEFT JOIN Examinations e
    ON e.student_id = s.student_id AND e.subject_name = sub.subject_name
GROUP BY s.student_id, s.student_name, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

---

### Worked Example 3: Managers with 5+ Reports

**Tables:** Employee (self-referencing — id, name, managerId)

| Step | Answer |
|---|---|
| Classify tables | Single table, self-referencing (managerId → id) |
| Output includes all entities? | No — only qualifying managers |
| Join type (Pass 1) | **Self JOIN** (Employee as manager × Employee as report) |
| Output shape | One row per qualifying manager → GROUP BY |
| Transformation needed | Filter groups by count threshold → HAVING |
| NULL handling | INNER self-join, no LEFT JOIN NULLs |
| **Combo Pattern** | **C: INNER JOIN + GROUP BY + HAVING** |

```sql
SELECT m.name
FROM Employee m
JOIN Employee r ON m.id = r.managerId
GROUP BY m.id, m.name
HAVING COUNT(*) >= 5;
```

<hr style="border: 3px solid black;">

<a id='7-efficiency-where-to-aggregate'></a>

## 7. Efficiency: Where to Aggregate — Before or After the Join?

One of the most important decisions in combo problems is **when** to aggregate.

### Option A — Aggregate After Joining

```
Dimension  ──LEFT JOIN──▶  Fact  ──GROUP BY──▶  Result
```

**Pros:** Simpler SQL, one pass, optimizer handles it well for small-medium data.

**Cons:** The join can explode the row count before aggregation.

### Option B — Aggregate Before Joining (Pre-aggregate)

```
Fact  ──GROUP BY──▶  Summary  ──LEFT JOIN──▶  Dimension  ──▶  Result
```

**Pros:** Reduces fact table to summary first, then join is on fewer rows.

**Cons:** More complex SQL, need COALESCE for NULLs.

### When to Use Which

| Scenario | Best Choice |
|---|---|
| Fact table is small/medium | Aggregate after (simpler) |
| Fact table has millions of rows | Pre-aggregate (faster join) |
| Multiple aggregations on same fact table | Pre-aggregate once in CTE, reuse |
| Simple COUNT or SUM | Aggregate after (COUNT handles NULLs naturally) |
| Complex conditional metric (rate, ratio) | Either works — pre-aggregate if readability matters |

<hr style="border: 3px solid black;">

<a id='9-interview-script'></a>

## 9. Interview Script for Combo Problems

Use this template when explaining your approach:

| Step | Say This |
|---|---|
| 1 | *"I see [N] tables — [X] is the entity list and [Y] is the event data."* |
| 2 | *"The output needs all entities, even those with no events, so I'll use a LEFT JOIN from [X] to [Y]."* |
| 3 | *"The output has one row per entity, so I need GROUP BY."* |
| 4 | *"For the metric, I'll use [AVG(CASE WHEN...) / COUNT(column) / etc.] because [reason]."* |
| 5 | *"I'm using COUNT(column) instead of COUNT(*) because the LEFT JOIN introduces NULLs for unmatched rows, and COUNT(*) would give 1 instead of 0."* |
| 6 | *"An alternative would be to pre-aggregate in a subquery and join the result, which would be equally efficient."* |

### The Power of Step 5

Explicitly calling out the NULL trap in an interview is a **strong signal** to the interviewer. It shows you understand how joins interact with aggregation — a common source of bugs in production SQL.

<hr style="border: 3px solid black;">

<a id='subquery-placement-in-combos'></a>

## Subquery Placement in Combo Problems

Many combo problems involve a subquery as one of the building blocks. After you've picked your JOIN (Pass 1) and your transformation (Pass 2), ask: **does any part of my calculation need a value from a different aggregation level?** If yes, you need a subquery.

### Subquery Placement Decision Tree

<div class="fc">
  <div class="fc-node fc-start">After Pass 1 (JOIN) and Pass 2 (Transform),<br/>does my formula need a value from a DIFFERENT<br/>aggregation level or a different table?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches">
    <div class="fc-branch">
      <div class="fc-label fc-no">NO</div>
      <div class="fc-node fc-good">No subquery needed<br/>Your JOIN + transform is complete</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-yes">YES</div>
      <div class="fc-node fc-start">What role does the subquery play?</div>
    </div>
  </div>
  <div class="fc-arrow">▼</div>
  <div class="fc-branches fc-three">
    <div class="fc-branch">
      <div class="fc-label fc-tag">DENOMINATOR</div>
      <div class="fc-node fc-warn">Provides a total or baseline for a ratio<br/><br/>Subquery in SELECT</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">FILTER</div>
      <div class="fc-node fc-warn">Provides a threshold to compare against<br/><br/>Subquery in WHERE</div>
    </div>
    <div class="fc-branch">
      <div class="fc-label fc-tag">PRE-AGGREGATE</div>
      <div class="fc-node fc-warn">One table must be summarized BEFORE the join<br/><br/>Subquery in FROM (or CTE)</div>
    </div>
  </div>
</div>

---

### How Subqueries Fit Into the Two-Pass Framework

| Combo Pattern | Pass 1 (JOIN) | Pass 2 (Transform) | Subquery Role | Placement |
|---|---|---|---|---|
| Percentage of total | No join needed | GROUP BY + scalar division | Denominator from another table | **SELECT** |
| Filter by aggregate threshold | JOIN two tables | GROUP BY | Computed cutoff value | **WHERE** |
| Join pre-aggregated summaries | INNER/LEFT JOIN | Compare or rank | Derived table with GROUP BY | **FROM** |
| Anti-pattern check with threshold | LEFT JOIN | COUNT + HAVING | Average from the full dataset | **WHERE** or **HAVING** |

> **Key insight:** In combo problems, subqueries most often appear when one piece of the puzzle operates at a **different grain** than the rest. If you're grouping by `contest_id` but need the total across ALL users, that mismatch in grain is what forces the subquery.

---

### Worked Example: Percentage of Total (SELECT subquery in a combo)

**Problem:** Find the percentage of total users registered in each contest.

```sql
SELECT r.contest_id,
       ROUND(
           (COUNT(DISTINCT r.user_id)::numeric
            / (SELECT COUNT(DISTINCT user_id) FROM Users)
           ) * 100, 2
       ) AS percentage
FROM Register AS r
GROUP BY r.contest_id
ORDER BY percentage DESC, r.contest_id ASC;
```

**Decomposition:**
- **Pass 1:** No join needed — Register table has all the grouping data
- **Pass 2:** GROUP BY `contest_id` + COUNT per group
- **Subquery:** The denominator (`total users`) comes from a different table at a different grain → scalar subquery in SELECT

**Why not a JOIN?** We don't need any columns from Users in the output — just a single count. A subquery is cleaner than joining and then aggregating.

---

### Before You Reach for a Subquery — Check If Conditional Aggregation Works

Many problems that look like they need a subquery can be solved more efficiently with `CASE WHEN` inside an aggregate function. This is called **conditional aggregation** and it keeps everything in a single `GROUP BY` pass.

**Worked Example — Queries Problem:**

*Task:* For each query_name, compute the quality (average of rating/position) and the poor query percentage (percentage of ratings below 3).

```sql
-- Approach 1: Subquery + JOIN (2 scans, more complex)
SELECT q.query_name,
       ROUND(SUM(q.rating::numeric / q.position) / COUNT(*), 2) AS quality,
       ROUND(MAX(p.n_poor::numeric) / COUNT(*) * 100, 2) AS poor_pct
FROM Queries q
LEFT JOIN (
    SELECT query_name, COUNT(*) AS n_poor
    FROM Queries WHERE rating < 3
    GROUP BY query_name
) p ON q.query_name = p.query_name
GROUP BY q.query_name;

-- Approach 2: Conditional aggregation (1 scan, cleaner)
SELECT query_name,
       ROUND(AVG(rating::numeric / position), 2) AS quality,
       ROUND(AVG(CASE WHEN rating < 3 THEN 100.0 ELSE 0 END), 2) AS poor_pct
FROM Queries
GROUP BY query_name;
```

**Decision checklist — can I skip the subquery?**

| Question | YES means skip | NO means you need it |
|---|---|---|
| Can the filter condition be checked row-by-row? | Use `CASE` inside aggregate | Need subquery for pre-filtering |
| Is the GROUP BY grain the same for all metrics? | Single GROUP BY works | Subquery aggregates at different grain |
| Does the denominator come from the same rows? | `AVG(CASE ...)` handles it | Subquery for separate denominator |

> **Pattern to memorize:** `AVG(CASE WHEN condition THEN 100.0 ELSE 0 END)` = percentage of rows meeting the condition. Works because averaging 0s and 100s gives the percent directly.

<hr style="border: 3px solid black;">

<a id='10-common-mistakes'></a>

## 10. Common Combo Mistakes

| Mistake | Why It's Wrong | Fix |
|---|---|---|
| Using INNER JOIN when output needs zeros | Drops entities with no events | Use LEFT JOIN |
| Using COUNT(*) after LEFT JOIN | Counts NULL rows as 1 | Use COUNT(right_table.column) |
| Forgetting COALESCE with SUM after LEFT JOIN | Returns NULL instead of 0 | Wrap in COALESCE(..., 0) |
| CROSS JOIN when only one dimension exists | Unnecessary Cartesian product | Use LEFT JOIN directly |
| Aggregating before joining when you need row-level data | Loses detail needed for window functions | Join first, then apply window function |
| Using CASE WHEN col = 'x' without ELSE after LEFT JOIN | NULL ≠ 'x' falls through without ELSE | Always include ELSE for the NULL case |
| Using NOT IN to find missing when NULLs possible | Silently returns 0 rows | Use LEFT JOIN + IS NULL or NOT EXISTS |

<hr style="border: 3px solid black;">

<a id='11-final-takeaway'></a>

## 11. Final Takeaway

Every combo problem breaks down the same way:

```
Pass 1: Pick the JOIN  →  Pass 2: Pick the TRANSFORM  →  Handle NULLs
```

### The 4 Most Common Combos

| # | Combo | Recipe |
|---|---|---|
| 1 | Metric per entity including zeros | LEFT JOIN + GROUP BY + AVG(CASE WHEN) |
| 2 | Count per entity including zeros | LEFT JOIN + GROUP BY + COUNT(column) |
| 3 | Entities meeting a threshold | JOIN + GROUP BY + HAVING |
| 4 | Metric per combination including zeros | CROSS JOIN + LEFT JOIN + GROUP BY + COUNT(column) |

### Memory Rule

> *"LEFT JOIN means NULLs. NULLs mean COUNT(column) not COUNT(*). If I need a rate, AVG(CASE WHEN) handles NULLs automatically."*

If you can decompose a problem into Pass 1 + Pass 2, recognize which combo pattern it matches, and handle the NULL trap correctly, you can solve any multi-table SQL interview question.